# Entrenamiento del modelo

### Importación de librerías

In [ ]:
import sys
sys.path.append("..")
from scripts.carga_modelo import cargar_modelo
import pandas as pd
import os
import mysql.connector
from dotenv import load_dotenv
import torch
from torch.utils.data import random_split
from peft import LoraConfig

### Rutas

In [23]:
datos_limpios = '../datos/datos_limpios.csv'
contrasenia = '../.env'
esquemas = '../esquemas/'

### Importación de datos

In [10]:
datos = pd.read_csv(datos_limpios)
datos = datos.drop(columns=['valida'])
datos.head(2)

,db_id,pregunta,sql
0,department_management,¿Cuántos jefes de departamento tienen más de 5...,SELECT count(*) FROM head WHERE age > 56
1,department_management,"Indique el nombre, estado de nacimiento y edad...","SELECT name, born_state, age FROM head ORDER..."


### Conexión a la base de datos

In [9]:
# Contraseña del proyecto
load_dotenv(contrasenia)

# Conexión con MySQL
conexion = mysql.connector.connect(host='mysql', port=3306, user='root',
                                   password=os.getenv('MYSQL_ROOT_PASSWORD'))

# Verificar la conexión
if conexion.is_connected():
    print('Conexión exitosa')

Conexión exitosa


### Separación de variables independientes de las dependientes

In [16]:
X = datos[['db_id', 'pregunta']]
y = datos['sql']

### Agregando el esquema a los datos

In [24]:
# Función para leer el esquema.
def obtener_esquema(id_esquema):
    ruta = esquemas +  f'{id_esquema}.sql'

    with open(ruta, 'r', encoding='utf-8') as archivo:
        return archivo.read()

X['esquema'] = X['db_id'].apply(obtener_esquema)
X = X.drop(columns=['db_id'])
X.head(2)

,pregunta,esquema
0,¿Cuántos jefes de departamento tienen más de 5...,CREATE TABLE `department` (`Department_ID` DEC...
1,"Indique el nombre, estado de nacimiento y edad...",CREATE TABLE `department` (`Department_ID` DEC...


### División del conjunto de entrenamiento y prueba

In [25]:
SEMILLA = 0
X_train, X_test = random_split(X, lengths=[0.8, 0.2], generator=torch.Generator().manual_seed(SEMILLA))
y_train, y_test = random_split(y, lengths=[0.8, 0.2], generator=torch.Generator().manual_seed(SEMILLA))

### Configuraciones de LoRA

Se realizará un cambio en el parámetro `r` (8 y 16).  

**Hipótesis**  
Aumentar el parámetro `r` incrementará la capacidad de adaptación del modelo y mejorará la generación de consultas SQL. La desventaja es que aumentará la cantidad de parámetros entrenables.

In [ ]:
modelo_1, tokenizer_1 = cargar_modelo(r=8)
modelo_2, tokenizer_2 = cargar_modelo(r=16)